# 02: Feature Engineering

Turns the cleaned candidate-race and party-list tables from `01` into one row per `(year, position, locality)` describing how the vote was distributed in that race: vote shares, margin between the top two finishers, and field concentration/fragmentation. No clustering here -- this builds the inputs for it.

**National vs. local races (carried over from `01`):** `SENATOR`, `PRESIDENT`, `VICE PRESIDENT` are reported per locality here, but the office is decided by the national vote sum. The `top_candidate`/`top1_share` columns below describe locally-most-voted, a useful spatial-clustering signal, not a claim about who won. For local offices the two usually coincide, though the Winners file's own gaps (`01`, ~8.6% of rows missing `City`) mean they can still differ.

Out of scope here: rolling per-race rows up to one row per locality (left to `03`, which knows how it wants to use them), joining across election years, and any PSGC/population/geographic join. Party-list gets the same descriptive share/concentration features as candidate races here, not an actual panachage-style seat-allocation model.

`build_race_features` (`src/common.py`) is shared with `01`'s locality-grouping convention.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..") / "src"))
from common import build_race_features

pd.set_option("display.max_columns", 30)
print("pandas:", pd.__version__)

pandas: 3.0.2


In [2]:
CONFIG = {
    "processed_dir": "../data/processed",
}

print("Config set.")

Config set.


## Load the cleaned tables from 01

In [3]:
processed_dir = Path(CONFIG["processed_dir"])

races = pd.read_parquet(processed_dir / "votes_races_clean.parquet")
partylist = pd.read_parquet(processed_dir / "votes_partylist_clean.parquet")

print("Candidate races:", races.shape)
print("Party list:", partylist.shape)

Candidate races: (932985, 21)
Party list: (1151975, 20)


## Restrict to real localities

`is_geographic` marks rows that belong to an actual province/city rather than an overseas or absentee-voting reporting post (e.g. "AMERICAS", "AGANA"), flagged in `01`. Dropped before building spatial features.

In [4]:
n_before = len(races)
races = races[races["is_geographic"]].copy()
print(f"Candidate races: kept {len(races):,} of {n_before:,} rows "
      f"({len(races) / n_before:.1%}) after dropping non-geographic reporting posts.")

n_before_pl = len(partylist)
partylist = partylist[partylist["is_geographic"]].copy()
print(f"Party list: kept {len(partylist):,} of {n_before_pl:,} rows "
      f"({len(partylist) / n_before_pl:.1%}).")

Candidate races: kept 930,844 of 932,985 rows (99.8%) after dropping non-geographic reporting posts.


Party list: kept 1,147,395 of 1,151,975 rows (99.6%).


## Race-level features

For each `(year, position, locality)` race:

- `n_candidates` -- how many names were on the ballot for that race.
- `total_votes` -- votes cast across all candidates in that race (not turnout: there's no registered-voter denominator in this data, so this is a votes-cast count, nothing more).
- `top1_share` / `top2_share` -- the vote share of the most- and second-most-voted candidate.
- `margin` = `top1_share - top2_share` -- how close the top of the field was. An uncontested race (one candidate, `n_candidates == 1`) has no second place, so `top2_share` is treated as 0 and margin comes out to 1.0, the maximum -- the least competitive case on the scale, not a missing-data artifact.
- `hhi` (Herfindahl-Hirschman Index) = sum of every candidate's share squared -- a standard concentration measure: 1.0 when one candidate takes all the votes, small when many candidates split the vote evenly.
- `enc` (effective number of candidates, Laakso-Taagepera) = `1 / hhi` -- the same information as `hhi` rescaled into "roughly this many candidates were seriously competing."

Multi-seat races (`COUNCILOR`, `PROVINCIAL BOARD MEMBER` locally; `SENATOR` nationally) elect more than one winner, so `top1`/`top2`/`margin` describe the gap between the single most- and second-most-voted candidates, not a seat-winning cutoff -- an actual cutoff needs the seat count, which depends on the Winners file `01` already cross-validated separately.

A handful of races have zero total votes recorded for every candidate; share, margin, and HHI are undefined (0/0) there, so `build_race_features` drops those groups and reports the count.

In [5]:
race_features, race_dropped = build_race_features(races, group_extra_cols=["region"])
if race_dropped is not None:
    print(f"Dropping {race_dropped['n_races'].sum():,} race(s) with zero total votes recorded "
          f"(share/margin/HHI are undefined when nobody's tally is known):")
    display(race_dropped)

print("Race features:", race_features.shape)
race_features.head()

Dropping 252 race(s) with zero total votes recorded (share/margin/HHI are undefined when nobody's tally is known):


,position,year,n_races
0,BARMM MEMBER OF PARLIAMENT,2025,102
1,BARMM PARTY REPRESENTATIVE,2025,102
2,COUNCILOR,2019,7
3,GOVERNOR,2019,7
4,MAYOR,2019,7
5,PROVINCIAL BOARD MEMBER,2019,7
6,SENATOR,2019,7
7,VICE GOVERNOR,2019,7
8,VICE MAYOR,2019,6


Race features: (75055, 15)


,year,province,city,n_candidates,total_votes,hhi,top_candidate,top_party,top1_share,top2_share,region,position,district,margin,enc
0,2016,BASILAN,AKBAR,6,14764,0.298454,"HANTIAN, RONIE",IND,0.323828,0.314210,BARMM,ARMM ASSEMBLYMAN,<NA>,0.009618,3.350605
1,2016,BASILAN,AL BARKA,6,18380,0.235216,"HANTIAN, RONIE",IND,0.315234,0.266703,BARMM,ARMM ASSEMBLYMAN,<NA>,0.048531,4.251408
2,2016,BASILAN,HADJI MOHAMMAD AJUL,6,16035,0.224304,"HANTIAN, RONIE",IND,0.287496,0.273714,BARMM,ARMM ASSEMBLYMAN,<NA>,0.013782,4.458233
3,2016,BASILAN,HADJI MUHTAMAD,6,12620,0.269859,"HANTIAN, RONIE",IND,0.317036,0.305864,BARMM,ARMM ASSEMBLYMAN,<NA>,0.011173,3.705635
4,2016,BASILAN,LAMITAN,6,83128,0.208637,"HANTIAN, RONIE",IND,0.266120,0.231378,BARMM,ARMM ASSEMBLYMAN,<NA>,0.034742,4.793007


## Party-list features

Party-list ballots are cast for the party itself, not a named candidate -- `candidate_name` and `party` are identical on every party-list row (517 distinct values in each). The same feature function applies directly, treating each party as the "candidate" in an otherwise identical per-locality race.

In [6]:
partylist_features, pl_dropped = build_race_features(partylist, group_extra_cols=["region"])
if pl_dropped is not None:
    print(f"Dropping {pl_dropped['n_races'].sum():,} race(s) with zero total votes recorded:")
    display(pl_dropped)

partylist_features = partylist_features.rename(
    columns={"top_candidate": "top_party_name"}
).drop(columns=["top_party"])
print("Party-list features:", partylist_features.shape)
partylist_features.head()

Dropping 7 race(s) with zero total votes recorded:


,position,year,n_races
0,PARTY LIST,2019,7


Party-list features: (7587, 13)


,year,province,city,n_candidates,total_votes,hhi,top_party_name,top1_share,top2_share,region,position,margin,enc
0,2010,ABRA,BANGUED,187,14717,0.072327,ASSOCIATION OF PHILIPPINE ELECTRIC COOPERATIVES,0.219066,0.106815,CORDILLERA ADMINISTRATIVE REGION,PARTY LIST,0.112251,13.826069
1,2010,ABRA,BOLINEY,187,1736,0.121056,"AGAPAY NG INDIGENOUS PEOPLES RIGHTS ALLIANCE, ...",0.314516,0.080645,CORDILLERA ADMINISTRATIVE REGION,PARTY LIST,0.233871,8.260639
2,2010,ABRA,BUCAY,187,6408,0.056164,ASSOCIATION OF PHILIPPINE ELECTRIC COOPERATIVES,0.148408,0.119226,CORDILLERA ADMINISTRATIVE REGION,PARTY LIST,0.029182,17.805109
3,2010,ABRA,BUCLOC,187,1112,0.094944,"AGILA NG KATUTUBONG PILIPINO, INC.",0.201439,0.161871,CORDILLERA ADMINISTRATIVE REGION,PARTY LIST,0.039568,10.532563
4,2010,ABRA,DAGUIOMAN,187,705,0.059601,COOPERATIVE NATCCO NETWORK PARTY,0.136170,0.120567,CORDILLERA ADMINISTRATIVE REGION,PARTY LIST,0.015603,16.778348


## Sanity checks

Range checks on share/HHI, plus a spot check against the Bangued, Abra 2010 Councilor race (inspected by hand in `01`) confirming feature numbers match the raw vote counts.

In [7]:
assert race_features["top1_share"].between(0, 1 + 1e-9).all(), "top1_share out of [0, 1]"
assert race_features["margin"].between(0, 1 + 1e-9).all(), "margin out of [0, 1]"
assert (race_features["hhi"] > 0).all() and (race_features["hhi"] <= 1 + 1e-9).all(), "hhi out of (0, 1]"
assert (race_features["enc"] >= 1 - 1e-9).all(), "enc should never be below 1"
assert (race_features["n_candidates"] >= 1).all()
n_uncontested = (race_features["n_candidates"] == 1).sum()
print(f"All range checks passed. {n_uncontested:,} of {len(race_features):,} local races "
      f"({n_uncontested / len(race_features):.1%}) were uncontested (one candidate).")

bangued = race_features[(race_features["year"] == 2010) & (race_features["province"] == "ABRA")
                         & (race_features["city"] == "BANGUED") & (race_features["position"] == "COUNCILOR")]
print("\nBangued, Abra, 2010 Councilor (top vote-getter was BORJA, SALVACION B. with 9,311 of the")
print("race's votes per the raw file, per candidate ahead of the runner-up ADAME by 100 votes):")
display(bangued[["top_candidate", "n_candidates", "total_votes", "top1_share", "top2_share", "margin", "enc"]])

All range checks passed. 5,338 of 75,055 local races (7.1%) were uncontested (one candidate).

Bangued, Abra, 2010 Councilor (top vote-getter was BORJA, SALVACION B. with 9,311 of the
race's votes per the raw file, per candidate ahead of the runner-up ADAME by 100 votes):


,top_candidate,n_candidates,total_votes,top1_share,top2_share,margin,enc
558,"BORJA, SALVACION B.",15,104836,0.088815,0.087861,0.000954,14.027948


## Save

In [8]:
race_features.to_parquet(processed_dir / "race_features.parquet", index=False)
partylist_features.to_parquet(processed_dir / "partylist_features.parquet", index=False)

print("Saved to", processed_dir.resolve())
print(f" - race_features.parquet       {race_features.shape}")
print(f" - partylist_features.parquet  {partylist_features.shape}")

Saved to /tmp/claude-0/-home-claude/8d12b1ae-8bb8-5630-9195-30e2920d12b4/scratchpad/election-spatial-analysis/data/processed
 - race_features.parquet       (75055, 15)
 - partylist_features.parquet  (7587, 13)


## Summary

`race_features.parquet` and `partylist_features.parquet` hold one row per `(year, position, locality)` with vote-share, margin, and concentration (HHI, effective number of candidates) features, built from vote counts directly -- no dependency on the Winners file, so its gaps don't propagate here.

**Next:** `03` rolls these per-race rows up into one row per locality per election (which positions to include, how to weight them). Temporal joins across years and the PSGC/population/geographic join are also still open.